# Build Corpora with sentence-corpora

This notebook builds and pickles the stylometric corpora using the sentence-corpora package.

## Stages

1. **Setup** - Import libraries and define paths
2. **Load Data** - Read masked files from data/
3. **Build TwoLevelCorpus** - Create Sentence objects with metadata
4. **Split Corpora** - Divide into train and test sets, holding out X_ordine
5. **Save Corpora** - Pickle corpora for later use

## Stage 1: Setup

Import required libraries and define paths.

In [33]:
from __future__ import annotations

import pickle
import random
import re
from collections import defaultdict
from pathlib import Path

from sentence_corpora import Sentence, TwoLevelCorpus

## Stage 2: Load Data

Read all masked files and extract metadata from filenames.

In [35]:
DATA_DIR: Final = Path("data_processed_ordine")


def parse_filename(path: Path) -> tuple[str, str]:
    """Parse '<authorcode>_<workcode>_sentences.quesplit.masked.txt' into (author, work).

    Handles extra whitespace and underscores by splitting on '_sentences.quesplit.masked.txt'.
    """
    name = path.name
    # Remove the _sentences.quesplit.masked.txt suffix
    name = re.sub(r"_sentences\.quesplit\.masked\.txt$", "", name)
    # Split on first underscore to separate authorcode and workcode
    parts = name.split("_", 1)
    if len(parts) != 2:
        raise ValueError(
            f"Expected '<authorcode>_<workcode>' in filename, got: {path.name!r}"
        )
    return parts[0], parts[1]


def build_corpus(data_dir: Path = DATA_DIR) -> TwoLevelCorpus:
    """Read all *_sentences.quesplit.masked.txt files and return a TwoLevelCorpus."""
    sentences = []
    masked_files = sorted(data_dir.glob("*_sentences.quesplit.masked.txt"))

    for path in masked_files:
        author, work = parse_filename(path)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    sentences.append(
                        Sentence(
                            text=line,
                            metadata={
                                "author": author,
                                "work": work,
                            },
                        )
                    )

    return TwoLevelCorpus(sentences, levels=["author", "work"])

In [36]:
corpus = build_corpus()
print(f"Built corpus with {len(corpus)} sentences")
print(f"Authors: {corpus.get_unique_values('author')}")
unique_works = set((s.metadata["author"], s.metadata["work"]) for s in corpus)
print(f"Works: {len(unique_works)} total")

for author, work in corpus.get_unique_tuples():
    print(f"{author}_{work}")

Built corpus with 47269 sentences
Authors: ['AI', 'AeC', 'AeP', 'B', 'BC', 'DC', 'E', 'FB', 'GAp', 'GAr', 'GC', 'GM', 'GP', 'GVS', 'GiP', 'H', 'HA', 'HP', 'IH', 'II', 'M', 'NLC', 'OM', 'OSE', 'RC', 'RQ', 'RT', 'SG', 'U1', 'U10', 'U11', 'U2', 'U3', 'U4', 'U5', 'U6', 'U7', 'U8', 'U9', 'X']
Works: 44 total
AI_anticlaudianus
AeC_medicaminum
AeC_viaticus
AeP_karolinus
B_judas
BC_contemptu
DC_mathildis
E_biblia
FB_nuptis
GAp_roberti
GAr_philippis
GC_alexandreis
GM_merlini
GP_ligurinus
GVS_poetria
GiP_historia
H_maria
HA_planctus
HP_maiolichinus
IH_architrenius
II_ylias
M_capitulorum
M_maurilii
M_victoris
NLC_passio
OM_ernestus
OSE_doctrina
RC_tancredi
RQ_catherinae
RT_mauri
SG_ecclesiasten
U1_canticamariae
U10_karolellus
U11_katherinae
U2_liberesdre
U3_generatione
U4_dialogi
U5_britanniae
U5_brutus
U6_roudlieb
U7_flosmedicinae
U8_friderici
U9_vetula
X_ordine


## Stage 3: Split Corpora

Split corpus into train and test sets, holding out the work X_ordine.

Returns (train, test, holdout) where:
  - train contains ~70% of tokens from every work except X_ordine (configurable via train_pct)
  - test contains the remaining ~30%
  - *De ordine* is held out completely (not used in train or test)

In [38]:
def train_test_split(
    corpus: TwoLevelCorpus, seed: int = 42, train_pct: float = 0.5
) -> tuple[TwoLevelCorpus, TwoLevelCorpus, TwoLevelCorpus]:
    """Split corpus into train, test, and a hold-out corpus for X_ordine.

    Args:
        corpus: The corpus to split.
        seed: Random seed for reproducibility.
        train_pct: Fraction of tokens to allocate to train (0-1). Test gets 1-train_pct.

    Returns (train, test, holdout) where:
      - holdout contains all sentences from X_ordine
      - train contains ~train_pct of tokens from every other work
      - test contains the remaining ~1-train_pct tokens
    """
    holdout_sentences = []
    test_sentences = []
    train_sentences = []

    rng = random.Random(seed)

    # Group sentences by (author, work)
    by_work = defaultdict(list)
    for s in corpus:
        key = (s.author, s.work)
        by_work[key].append(s)

    for (author, work), sentences in by_work.items():
        if work == "ordine" and author == "X":
            holdout_sentences.extend(sentences)
            continue
        # Shuffle, then greedily add sentences to train until ~train_pct of tokens reached
        shuffled = list(sentences)
        rng.shuffle(shuffled)

        total_tokens = sum(len(s.text.split()) for s in shuffled)
        target = total_tokens * train_pct

        accumulated = 0
        cutoff = len(shuffled)
        for i, s in enumerate(shuffled):
            if accumulated >= target:
                cutoff = i
                break
            accumulated += len(s.text.split())

        train_sentences.extend(shuffled[:cutoff])
        test_sentences.extend(shuffled[cutoff:])

    return (
        TwoLevelCorpus(train_sentences, levels=["author", "work"]),
        TwoLevelCorpus(test_sentences, levels=["author", "work"]),
        TwoLevelCorpus(holdout_sentences, levels=["author", "work"]),
    )

In [39]:
train_corpus, test_corpus, ordine_corpus = train_test_split(corpus, train_pct=0.7)
print(f"Train corpus: {len(train_corpus)} sentences")
print(f"Test corpus:  {len(test_corpus)} sentences")
print(f"Holdout corpus (X_ordine): {len(ordine_corpus)} sentences")
print()

Train corpus: 32908 sentences
Test corpus:  14080 sentences
Holdout corpus (X_ordine): 281 sentences



In [40]:
train_corpus.stats()

Author      Works    Sentences    Tokens    Mean Len
--------  -------  -----------  --------  ----------
AI              1        1,076    23,027        21.4
AeC             2        1,183    34,577        29.2
AeP             1          439    13,807        31.5
B               1          290     4,714        16.3
BC              1        1,890    16,325         8.6
DC              1          895    15,050        16.8
E               1          351     7,814        22.3
FB              1        1,882    24,966        13.3
GAp             1          952    14,010        14.7
GAr             1        1,264    26,546        21
GC              1        1,457    28,398        19.5
GM              1          452     7,959        17.6
GP              1        1,208    33,758        27.9
GVS             1          942    11,604        12.3
GiP             1          732    13,405        18.3
H               1          293     4,631        15.8
HA              1          872    11,812        

In [41]:
test_corpus.stats()

Author      Works    Sentences    Tokens    Mean Len
--------  -------  -----------  --------  ----------
AI              1          470     9,861        21
AeC             2          509    14,745        29
AeP             1          190     5,806        30.6
B               1          111     1,991        17.9
BC              1          830     6,994         8.4
DC              1          398     6,449        16.2
E               1          160     3,326        20.8
FB              1          775    10,684        13.8
GAp             1          423     6,004        14.2
GAr             1          532    11,367        21.4
GC              1          612    12,123        19.8
GM              1          184     3,379        18.4
GP              1          516    14,417        27.9
GVS             1          400     4,958        12.4
GiP             1          327     5,741        17.6
H               1          117     1,960        16.8
HA              1          372     5,049        13

In [42]:
train_1000 = train_corpus.chunk(1000)
train_1000.stats()

Author      Works    Sentences    Tokens    Mean Len
--------  -------  -----------  --------  ----------
AI             22        1,076    23,027        21.4
AeC            33        1,183    34,577        29.2
AeP            13          439    13,807        31.5
B               4          290     4,714        16.3
BC             16        1,890    16,325         8.6
DC             14          895    15,050        16.8
E               7          351     7,814        22.3
FB             24        1,882    24,966        13.3
GAp            13          952    14,010        14.7
GAr            26        1,264    26,546        21
GC             28        1,457    28,398        19.5
GM              7          452     7,959        17.6
GP             33        1,208    33,758        27.9
GVS            11          942    11,604        12.3
GiP            13          732    13,405        18.3
H               4          293     4,631        15.8
HA             11          872    11,812        

In [43]:
# Check authors with fewer than 5000 total tokens to make sure the splitting is working as expected
for author in train_1000.get_unique_values("author"):
    author_corpus = train_1000.filter_by_level("author", author)
    author_tokens = sum(len(s.text.split()) for s in author_corpus)
    if author_tokens < 5000:
        print(f"\n{author}: {author_tokens} tokens")
        for work_name in author_corpus.get_unique_values("work"):
            work_corpus = author_corpus.filter_by_level("work", work_name)
            work_tokens = sum(len(s.text.split()) for s in work_corpus)
            print(f"  {work_name}: {work_tokens} tokens")


B: 4714 tokens
  judas_1: 1168 tokens
  judas_2: 1188 tokens
  judas_3: 1182 tokens
  judas_4: 1176 tokens

H: 4631 tokens
  maria_1: 1151 tokens
  maria_2: 1154 tokens
  maria_3: 1167 tokens
  maria_4: 1159 tokens

RT: 4446 tokens
  mauri_1: 1095 tokens
  mauri_2: 1109 tokens
  mauri_3: 1122 tokens
  mauri_4: 1120 tokens

SG: 1706 tokens
  ecclesiasten_1: 1706 tokens

U1: 1622 tokens
  canticamariae_1: 1622 tokens

U2: 2243 tokens
  liberesdre_1: 1126 tokens
  liberesdre_2: 1117 tokens


## Stage 4: Save Corpora

Pickle corpora for later use.

In [45]:
with open("train_corpus.pkl", "wb") as f:
    pickle.dump(train_corpus, f)

with open("test_corpus.pkl", "wb") as f:
    pickle.dump(test_corpus, f)

with open("ordine_corpus.pkl", "wb") as f:
    pickle.dump(ordine_corpus, f)

print("Corpora pickled successfully.")
print("Files created: train_corpus.pkl, test_corpus.pkl, ordine_corpus.pkl")

Corpora pickled successfully.
Files created: train_corpus.pkl, test_corpus.pkl, ordine_corpus.pkl
